In [102]:
!pip install -q pyspark

In [103]:
import os

# Create Data Lake folder structure
os.makedirs("/content/data_lake/bronze", exist_ok=True)
os.makedirs("/content/data_lake/silver/preprocessed_sales", exist_ok=True)

print("Data Lake structure created successfully!")

Data Lake structure created successfully!


In [104]:
import shutil

shutil.copy(
    "/content/sales.csv",
    "/content/data_lake/bronze/sales.csv"
)

print("Original sales.csv copied to Bronze.")

Original sales.csv copied to Bronze.


In [105]:
import os

print(os.path.exists("/content/data_lake/bronze/sales.csv"))

True


In [106]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("SalesDataLake") \
    .getOrCreate()

print("SparkSession created successfully!")

SparkSession created successfully!


In [107]:
bronze_path = "/content/data_lake/bronze/sales.csv"

df = spark.read.csv(
    bronze_path,
    header=True,
    inferSchema=True
)

print("Sales data loaded successfully!")

Sales data loaded successfully!


In [108]:
df.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)



In [109]:
df.show(5, truncate=False)

+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name |product      |category|quantity|unit_price|discount_percent|payment_method|city      |state         |order_date|order_status|
+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|100001  |C0256      |Keerthi Mishra|Saree        |Clothing|2       |6710      |20              |Credit Card   |Mumbai    |Maharashtra   |2026-01-01|Delivered   |
|100002  |C0042      |Arjun Singh   |T-Shirt      |Clothing|6       |680       |5               |Debit Card    |Chennai   |Tamil Nadu    |2025-08-03|Delivered   |
|100003  |C0348      |Keerthi Rao   |Hoodie       |Clothing|6       |3080      |12              |UPI           |Pune      |Maharashtra   |2025-05-06|Shipped     |
|100004  |C0051      |

In [110]:
print("Total records:", df.count())


Total records: 1000


In [111]:
# Check exact duplicate rows

total_rows = df.count()
distinct_rows = df.dropDuplicates().count()
duplicate_rows = total_rows - distinct_rows

print("Total rows:", total_rows)
print("Distinct rows:", distinct_rows)
print("Duplicate rows:", duplicate_rows)

Total rows: 1000
Distinct rows: 990
Duplicate rows: 10


In [112]:
from pyspark.sql.functions import col, sum

null_counts = df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])

null_counts.show(truncate=False)

+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|order_id|customer_id|customer_name|product|category|quantity|unit_price|discount_percent|payment_method|city|state|order_date|order_status|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|0       |0          |10           |0      |0       |0       |0         |0               |8             |8   |0    |0         |0           |
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+



In [113]:
print("Columns:")
for column in df.columns:
    print("-", column)

print("\nData types:")
df.dtypes

Columns:
- order_id
- customer_id
- customer_name
- product
- category
- quantity
- unit_price
- discount_percent
- payment_method
- city
- state
- order_date
- order_status

Data types:


[('order_id', 'int'),
 ('customer_id', 'string'),
 ('customer_name', 'string'),
 ('product', 'string'),
 ('category', 'string'),
 ('quantity', 'int'),
 ('unit_price', 'int'),
 ('discount_percent', 'int'),
 ('payment_method', 'string'),
 ('city', 'string'),
 ('state', 'string'),
 ('order_date', 'string'),
 ('order_status', 'string')]

In [114]:
from pyspark.sql.types import StringType

string_columns = [
    field.name
    for field in df.schema.fields
    if isinstance(field.dataType, StringType)
]

print("Text columns:", string_columns)

for c in string_columns:
    print(f"\nUnique values in {c}:")
    df.select(c).distinct().show(20, truncate=False)

Text columns: ['customer_id', 'customer_name', 'product', 'category', 'payment_method', 'city', 'state', 'order_date', 'order_status']

Unique values in customer_id:
+-----------+
|customer_id|
+-----------+
|C0114      |
|C0141      |
|C0243      |
|C0034      |
|C0154      |
|C0347      |
|C0130      |
|C0338      |
|C0167      |
|C0179      |
|C0195      |
|C0150      |
|C0033      |
|C0107      |
|C0016      |
|C0131      |
|C0191      |
|C0261      |
|C0057      |
|C0280      |
+-----------+
only showing top 20 rows

Unique values in customer_name:
+------------------+
|customer_name     |
+------------------+
|Pranav Jain       |
|Tanvi Rao         |
|Meera Pandey      |
|Tanvi Patel       |
|Rahul Jain        |
|Kavya Bose        |
|Siddharth Menon   |
|Divya Naidu       |
|Aditi Agarwal     |
|Aman Naidu        |
|Siddharth Malhotra|
|Sneha Chopra      |
|Vikram Reddy      |
|Harsh Iyer        |
|Tanvi Naidu       |
|Swathi Gupta      |
|Shreya Gupta      |
|Harsh Pillai      |

In [115]:
numeric_columns = [
    field.name
    for field in df.schema.fields
    if str(field.dataType) in [
        "IntegerType",
        "LongType",
        "DoubleType",
        "FloatType",
        "DecimalType"
    ]
]

print("Numeric columns:", numeric_columns)

df.select(numeric_columns).describe().show()

Numeric columns: []
+-------+
|summary|
+-------+
|  count|
|   mean|
| stddev|
|    min|
|    max|
+-------+



In [116]:
df.filter(col("Quantity") <= 0).show()

+--------+-----------+-------------+--------------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name|             product|       category|quantity|unit_price|discount_percent|  payment_method|      city|         state|order_date|order_status|
+--------+-----------+-------------+--------------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|  100114|      C0252|Keerthi Singh|         Ceiling Fan|Home Appliances|       0|      2920|              15|     Net Banking|   Chennai|    Tamil Nadu|2025-02-15|   Delivered|
|  100143|      C0184|   Aman Joshi|            Football|         Sports|       0|      1030|              15|      Debit Card|Coimbatore|    Tamil Nadu|2025-11-26|   Delivered|
|  100198|      C0014| Arjun Pillai|              Hoodie|       Clothing|      -2|      2160|              12|

In [117]:
df.filter(col("unit_price") < 0).show()

+--------+-----------+----------------+------------+-----------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|order_id|customer_id|   customer_name|     product|   category|quantity|unit_price|discount_percent|payment_method|      city|         state|order_date|order_status|
+--------+-----------+----------------+------------+-----------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|  100235|      C0349|     Tanvi Reddy|  Power Bank|Electronics|       7|      -500|               0|           UPI|Vijayawada|Andhra Pradesh|2026-03-02|   Delivered|
|  100284|      C0302|    Ananya Reddy| Study Table|  Furniture|       2|      -500|              30|    Debit Card|Vijayawada|Andhra Pradesh|2026-06-22|   Delivered|
|  100487|      C0232|Keerthi Kulkarni|Office Chair|  Furniture|       4|      -500|              25|    Debit Card|    Mumbai|   Maharashtra|2025-02-05|    Returned

In [118]:
df.filter(
    (col("discount_percent") < 0) | (col("discount_percent") > 100)
).show()


+--------+-----------+---------------+----------------+---------------+--------+----------+----------------+----------------+----------+----------+----------+------------+
|order_id|customer_id|  customer_name|         product|       category|quantity|unit_price|discount_percent|  payment_method|      city|     state|order_date|order_status|
+--------+-----------+---------------+----------------+---------------+--------+----------+----------------+----------------+----------+----------+----------+------------+
|  100034|      C0192|   Divya Sharma|        Yoga Mat|         Sports|       7|      1720|              -5|Cash on Delivery|Coimbatore|Tamil Nadu|2025-08-19|   Delivered|
|  100140|      C0075|Swathi Tripathi|  Cooking Oil 5L|        Grocery|       4|       790|              -5|     Credit Card| Hyderabad| Telangana|2025-10-01|     Shipped|
|  100217|      C0214|  Neha Tripathi|        Football|         Sports|       1|       650|              -5|             UPI|   Chennai|Tami

In [119]:

df.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)



In [120]:
df.select("Order_Date").distinct().show(30, truncate=False)

+----------+
|Order_Date|
+----------+
|2026-02-01|
|2025-06-03|
|2025-02-25|
|2026-02-27|
|2025-08-17|
|2026-07-23|
|2026-07-04|
|2025-09-14|
|2025-03-20|
|2025-04-05|
|2026-04-15|
|2026-06-05|
|2025-01-03|
|2025-12-07|
|2026-03-15|
|2025-12-30|
|2025-08-18|
|2026-09-11|
|2025-06-16|
|2026-09-06|
|2025-07-15|
|2026-08-24|
|2025/07/14|
|2025-05-06|
|2025-04-12|
|2026-02-15|
|2025-07-21|
|2025-05-23|
|2025-10-09|
|2026-01-05|
+----------+
only showing top 30 rows


In [121]:
df_clean = df

In [122]:
before_count = df_clean.count()

print("Row count before preprocessing:", before_count)

Row count before preprocessing: 1000


In [123]:
df_clean = df_clean.dropDuplicates()

after_duplicates = df_clean.count()

print("Rows after removing duplicates:", after_duplicates)
print("Duplicates removed:", before_count - after_duplicates)

Rows after removing duplicates: 990
Duplicates removed: 10


In [124]:
from pyspark.sql.functions import trim, initcap, lower

# Columns where normal capitalization is appropriate
title_case_columns = [
    "customer_name",
    "product",
    "category",
    "city",
    "state"
]

for c in title_case_columns:
    df_clean = df_clean.withColumn(
        c,
        initcap(trim(col(c)))
    )

# Clean payment method and status
df_clean = df_clean.withColumn(
    "payment_method",
    lower(trim(col("payment_method")))
)

df_clean = df_clean.withColumn(
    "order_status",
    lower(trim(col("order_status")))
)

In [125]:
text_columns = [
    "customer_name",
    "product",
    "category",
    "payment_method",
    "city",
    "state",
    "order_status"
]

for c in text_columns:
    df_clean = df_clean.fillna({c: "Unknown"})

In [126]:
from pyspark.sql.functions import when

df_clean = df_clean.withColumn(
    "quantity",
    when(col("quantity") > 0, col("quantity"))
    .otherwise(None)
)

In [127]:
df_clean = df_clean.withColumn(
    "unit_price",
    when(col("unit_price") >= 0, col("unit_price"))
    .otherwise(None)
)

In [128]:
df_clean = df_clean.withColumn(
    "discount_percent",
    when(
        (col("discount_percent") >= 0) &
        (col("discount_percent") <= 100),
        col("discount_percent")
    ).otherwise(None)
)

In [129]:
quantity_median = df_clean.approxQuantile(
    "quantity", [0.5], 0.0
)[0]

unit_price_median = df_clean.approxQuantile(
    "unit_price", [0.5], 0.0
)[0]

print("Quantity median:", quantity_median)
print("Unit price median:", unit_price_median)

Quantity median: 5.0
Unit price median: 1830.0


In [130]:
df_clean = df_clean.fillna({
    "quantity": quantity_median,
    "unit_price": unit_price_median,
    "discount_percent": 0
})

In [131]:
from pyspark.sql.functions import (
    col, trim, initcap, lower, when,
    coalesce, try_to_timestamp, lit
)

# Start again from the original Bronze DataFrame
df_clean = df.dropDuplicates()

# -----------------------------
# 1. Clean text columns
# -----------------------------
title_case_columns = [
    "customer_name",
    "product",
    "category",
    "city",
    "state"
]

for c in title_case_columns:
    df_clean = df_clean.withColumn(
        c,
        initcap(trim(col(c)))
    )

df_clean = df_clean.withColumn(
    "payment_method",
    lower(trim(col("payment_method")))
)

df_clean = df_clean.withColumn(
    "order_status",
    lower(trim(col("order_status")))
)

# -----------------------------
# 2. Handle missing text values
# -----------------------------
text_columns = [
    "customer_name",
    "product",
    "category",
    "payment_method",
    "city",
    "state",
    "order_status"
]

for c in text_columns:
    df_clean = df_clean.fillna({c: "Unknown"})

# -----------------------------
# 3. Validate quantity
# -----------------------------
df_clean = df_clean.withColumn(
    "quantity",
    when(col("quantity") > 0, col("quantity"))
    .otherwise(None)
)

# -----------------------------
# 4. Validate unit price
# -----------------------------
df_clean = df_clean.withColumn(
    "unit_price",
    when(col("unit_price") >= 0, col("unit_price"))
    .otherwise(None)
)

# -----------------------------
# 5. Validate discount
# -----------------------------
df_clean = df_clean.withColumn(
    "discount_percent",
    when(
        (col("discount_percent") >= 0) &
        (col("discount_percent") <= 100),
        col("discount_percent")
    ).otherwise(None)
)

# -----------------------------
# 6. Handle missing numerical values
# -----------------------------
quantity_median = df_clean.approxQuantile(
    "quantity", [0.5], 0.0
)[0]

unit_price_median = df_clean.approxQuantile(
    "unit_price", [0.5], 0.0
)[0]

df_clean = df_clean.fillna({
    "quantity": quantity_median,
    "unit_price": unit_price_median,
    "discount_percent": 0
})

# -----------------------------
# 7. Safely convert dates
# -----------------------------
df_clean = df_clean.withColumn(
    "order_date",
    coalesce(
        try_to_timestamp(
            col("order_date"), lit("yyyy-MM-dd")
        ),
        try_to_timestamp(
            col("order_date"), lit("dd/MM/yyyy")
        ),
        try_to_timestamp(
            col("order_date"), lit("MM/dd/yyyy")
        ),
        try_to_timestamp(
            col("order_date"), lit("dd-MM-yyyy")
        ),
        try_to_timestamp(
            col("order_date"), lit("MM-dd-yyyy")
        ),
        try_to_timestamp(
            col("order_date"), lit("yyyy/MM/dd")
        )
    ).cast("date")
)

print("Preprocessing completed successfully!")

Preprocessing completed successfully!


In [132]:
print("Checking invalid/missing dates:")

df_clean.filter(
    col("order_date").isNull()
).show(20, truncate=False)

Checking invalid/missing dates:
+--------+-----------+-------------+-------------+---------+--------+----------+----------------+----------------+---------+-------------+----------+------------+
|order_id|customer_id|customer_name|product      |category |quantity|unit_price|discount_percent|payment_method  |city     |state        |order_date|order_status|
+--------+-----------+-------------+-------------+---------+--------+----------+----------------+----------------+---------+-------------+----------+------------+
|100097  |C0244      |Shreya Rao   |Sofa         |Furniture|4       |47430     |10              |net banking     |Hyderabad|Telangana    |NULL      |returned    |
|100637  |C0020      |Aarav Patel  |Novel        |Books    |4       |610       |20              |net banking     |Chennai  |Tamil Nadu   |NULL      |delivered   |
|100448  |C0270      |Pranav Iyer  |Bedside Table|Furniture|1       |4290      |35              |upi             |Lucknow  |Uttar Pradesh|NULL      |deli

In [133]:
df_clean.show(10, truncate=False)

+--------+-----------+------------------+----------------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name     |product               |category       |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|
+--------+-----------+------------------+----------------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|100331  |C0005      |Sakshi Joshi      |Football              |Sports         |2       |670       |30              |credit card     |Bengaluru |Karnataka     |2025-06-06|cancelled   |
|100333  |C0263      |Siddharth Kulkarni|Programming Book      |Books          |4       |670       |0               |net banking     |Pune      |Maharashtra   |2025-05-08|delivered   |
|100506  |C0228      |Diya Tripathi     |Competitive Exam Guide|Books      

In [134]:
df_clean.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = false)
 |-- product: string (nullable = false)
 |-- category: string (nullable = false)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = false)
 |-- payment_method: string (nullable = false)
 |-- city: string (nullable = false)
 |-- state: string (nullable = false)
 |-- order_date: date (nullable = true)
 |-- order_status: string (nullable = false)



In [135]:
after_count = df_clean.count()

print("Row count before preprocessing:", before_count)
print("Row count after preprocessing:", after_count)

Row count before preprocessing: 1000
Row count after preprocessing: 990


In [136]:
silver_path = "/content/data_lake/silver/preprocessed_sales"

df_clean.coalesce(1).write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(silver_path)

print("Silver data saved successfully!")

Silver data saved successfully!


In [137]:
silver_df = spark.read.csv(
    silver_path,
    header=True,
    inferSchema=True
)

print("Silver data loaded successfully!")

Silver data loaded successfully!


In [138]:
silver_df.show(5, truncate=False)

+--------+-----------+------------------+----------------------+---------+--------+----------+----------------+----------------+----------+-----------+----------+------------+
|order_id|customer_id|customer_name     |product               |category |quantity|unit_price|discount_percent|payment_method  |city      |state      |order_date|order_status|
+--------+-----------+------------------+----------------------+---------+--------+----------+----------------+----------------+----------+-----------+----------+------------+
|100331  |C0005      |Sakshi Joshi      |Football              |Sports   |2       |670       |30              |credit card     |Bengaluru |Karnataka  |2025-06-06|cancelled   |
|100333  |C0263      |Siddharth Kulkarni|Programming Book      |Books    |4       |670       |0               |net banking     |Pune      |Maharashtra|2025-05-08|delivered   |
|100506  |C0228      |Diya Tripathi     |Competitive Exam Guide|Books    |6       |610       |12              |upi      

In [139]:
print("Silver row count:", silver_df.count())

Silver row count: 990


In [140]:
after_count = df_clean.count()

print("Row count before preprocessing:", before_count)
print("Row count after preprocessing:", after_count)
print("Rows removed:", before_count - after_count)

Row count before preprocessing: 1000
Row count after preprocessing: 990
Rows removed: 10


In [141]:
import os
import shutil

# Temporary Spark output folder
silver_temp = "/content/data_lake/silver/preprocessed_sales_temp"

# Remove old temporary output if it exists
if os.path.exists(silver_temp):
    shutil.rmtree(silver_temp)

# Write Silver data
df_clean.coalesce(1).write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(silver_temp)

# Find the generated CSV file
csv_file = [
    f for f in os.listdir(silver_temp)
    if f.endswith(".csv")
][0]

# Final required filename
final_silver = "/content/data_lake/silver/sales_silver.csv"

shutil.copy(
    os.path.join(silver_temp, csv_file),
    final_silver
)

print("Silver file created successfully!")
print(final_silver)

Silver file created successfully!
/content/data_lake/silver/sales_silver.csv


In [142]:
import os

print(os.path.exists("/content/data_lake/silver/sales_silver.csv"))

True


In [143]:
silver_df = spark.read.csv(
    "/content/data_lake/silver/sales_silver.csv",
    header=True,
    inferSchema=True
)

print("Silver data loaded successfully!")

Silver data loaded successfully!


In [144]:
silver_df.show(5, truncate=False)

+--------+-----------+------------------+----------------------+---------+--------+----------+----------------+----------------+----------+-----------+----------+------------+
|order_id|customer_id|customer_name     |product               |category |quantity|unit_price|discount_percent|payment_method  |city      |state      |order_date|order_status|
+--------+-----------+------------------+----------------------+---------+--------+----------+----------------+----------------+----------+-----------+----------+------------+
|100331  |C0005      |Sakshi Joshi      |Football              |Sports   |2       |670       |30              |credit card     |Bengaluru |Karnataka  |2025-06-06|cancelled   |
|100333  |C0263      |Siddharth Kulkarni|Programming Book      |Books    |4       |670       |0               |net banking     |Pune      |Maharashtra|2025-05-08|delivered   |
|100506  |C0228      |Diya Tripathi     |Competitive Exam Guide|Books    |6       |610       |12              |upi      

In [145]:
print("Silver row count:", silver_df.count())

Silver row count: 990


Preprocessing Performed
- Loaded the original sales dataset into the Bronze layer without modifying the raw data.
- Removed exact duplicate rows.
- Handled missing text values using "Unknown".
- Trimmed extra spaces and standardized capitalization in text fields.
- Validated quantity to ensure it is greater than zero.
- Validated unit_price to ensure it is non-negative.
- Validated discount_percent to ensure it is between 0 and 100.
- Handled invalid numerical values while preserving useful records.
- Converted valid order_date values to a consistent date format.
- Converted clearly invalid dates to null instead of removing the entire record.
- Saved the processed dataset as sales_silver.csv in the Silver layer.